### **Challenge 4** — Human-in-the-Loop Approval Agent

### What to build:
A LangGraph agent that proposes a refund action, pauses for human approval using interrupt(), then either executes or rejects based on human input.

### What it should do:

- User describes a refund scenario
- Agent proposes a specific action ("Refund $X for customer Y")
- Graph pauses — asks human "Approve or reject?"
- Human types "approve" → agent executes and confirms
- Human types "reject" → agent cancels and explains

### Constraints:

- Use interrupt() and Command(resume=...) — not plain input()
- Must use MemorySaver as checkpointer
- Same thread_id must be used for both the first invoke and the resume invoke
- Use TypedDict for state with at least: query, proposed_action, approval, result

In [5]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

True

In [6]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.graph.message import add_messages
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage
from langgraph.store.memory import InMemoryStore
import uuid # for thread_id

In [7]:
model = init_chat_model(
    model="openai/gpt-oss-120b",       # The specific Groq model ID
    model_provider="groq",        # Specifies the provider
    temperature=0                 # Optional parameters
)

In [ ]:
# Define State with required fields: query, proposed_action, approval, result
from typing import Literal
from typing_extensions import TypedDict

class State(TypedDict):
    query: str
    proposed_action: str
    approval: Literal["approve", "reject", "pending"]
    result: str

In [ ]:
# Build the Human-in-the-Loop Approval Graph
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.types import interrupt
from langchain_core.messages import HumanMessage

# Node 1: Analyze user query and propose a refund action
def propose_refund(state: State) -> State:
    """Analyze the refund request and propose a specific action."""
    user_query = state["query"]
    
    # Use LLM to parse the query and create a specific refund proposal
    prompt = f"""A user wants to request a refund. Their message: "{user_query}"
    
Extract the key details and propose a specific refund action in this format:
"Refund $X for customer Y (reason: Z)"

If details are missing, make reasonable assumptions but note them."""
    
    response = model.invoke([HumanMessage(content=prompt)])
    proposed = response.content
    
    return {
        **state,
        "proposed_action": proposed,
        "approval": "pending"
    }

# Node 2: Human approval via interrupt (pauses graph execution)
def human_approval(state: State) -> State:
    """Pause the graph and ask for human approval via interrupt()."""
    # This will pause execution and wait for human input via Command(resume=...)
    approval = interrupt({
        "proposed_action": state["proposed_action"],
        "question": "Approve or reject this refund action? (type 'approve' or 'reject')"
    })
    
    # approval will be the value passed via Command(resume=...)
    return {
        **state,
        "approval": approval
    }

# Node 3: Execute refund if approved, otherwise cancel
def execute_refund(state: State) -> State:
    """Execute the refund action based on approval."""
    if state["approval"] == "approve":
        result = f"✅ REFUND EXECUTED: {state['proposed_action']}"
    else:
        result = f"❌ REFUND CANCELLED: {state['proposed_action']} — Reason: Human rejected the action"
    
    return {
        **state,
        "result": result
    }

# Build the graph
builder = StateGraph(State)

builder.add_node("propose_refund", propose_refund)
builder.add_node("human_approval", human_approval)
builder.add_node("execute_refund", execute_refund)

builder.add_edge(START, "propose_refund")
builder.add_edge("propose_refund", "human_approval")
builder.add_edge("human_approval", "execute_refund")
builder.add_edge("execute_refund", END)

# Use MemorySaver as checkpointer (required by challenge)
checkpoint = MemorySaver()
store = InMemoryStore()

graph = builder.compile(checkpointer=checkpoint, store=store)

print("Graph compiled successfully!")
print("Nodes:", list(graph.nodes.keys()))

### Demo: Complete Human-in-the-Loop Flow

# Generate a thread_id to use for both invoke and resume (required by challenge)
thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

print(f"Thread ID: {thread_id}")
print("-" * 50)

# Step 1: First invoke - user describes refund scenario
user_query = "Customer John Smith wants a refund of $150 for order #12345 because the item arrived damaged"

initial_state = {
    "query": user_query,
    "proposed_action": "",
    "approval": "pending",
    "result": ""
}

print("📝 STEP 1: First invoke - Agent proposes refund action")
print(f"User query: {user_query}")
print()

# This will run until the interrupt() in human_approval node
result = graph.invoke(initial_state, config=config)

print(f"Proposed action: {result.get('proposed_action', 'N/A')}")
print(f"Status: Waiting for human approval...")
print()

# The graph is now paused at interrupt. We need to resume with approval.
# Step 2: Resume with human approval
print("📝 STEP 2: Resume with human approval ('approve')")
print("Human types: approve")
print()

# Resume the graph with approval
resume_result = graph.invoke(Command(resume="approve"), config=config)

print(f"Final result: {resume_result.get('result', 'N/A')}")
print()

print("=" * 50)
print("✅ Complete flow demonstrated!")
print("- Same thread_id used for both invoke and resume")
print("- interrupt() paused the graph")
print("- Command(resume=...) continued execution")
print("- MemorySaver checkpointer maintained state")

In [ ]:
### Demo 2: Reject Path

# New thread_id for a fresh conversation
thread_id_2 = str(uuid.uuid4())
config_2 = {"configurable": {"thread_id": thread_id_2}}

print(f"Thread ID: {thread_id_2}")
print("-" * 50)

user_query_2 = "Customer Jane Doe requests $75 refund for order #67890 - changed mind about purchase"

initial_state_2 = {
    "query": user_query_2,
    "proposed_action": "",
    "approval": "pending",
    "result": ""
}

print("📝 STEP 1: First invoke - Agent proposes refund action")
print(f"User query: {user_query_2}")
print()

result_2 = graph.invoke(initial_state_2, config=config_2)

print(f"Proposed action: {result_2.get('proposed_action', 'N/A')}")
print(f"Status: Waiting for human approval...")
print()

# Step 2: Resume with human REJECTION
print("📝 STEP 2: Resume with human rejection ('reject')")
print("Human types: reject")
print()

resume_result_2 = graph.invoke(Command(resume="reject"), config=config_2)

print(f"Final result: {resume_result_2.get('result', 'N/A')}")
print()

print("=" * 50)
print("✅ Reject path demonstrated!")